# Spill the Beans Order Portal

A backend portal to manage cafe orders.

- Orders are stored in a plain text file.
- Staff must log in before using the menu.

Each order has 5 fields:

- Order ID
- Customer name
- Item name
- Quantity
- Total price

## 1. Data file name

- DATA_FILE holds the name of the text file.
- Every function uses this one variable.
- To use a different file, change it here only.

In [1]:
DATA_FILE = "orders_data.txt"

## 2. load_data(): read orders from file

Runs once at startup and returns a list of orders.

- Opens the file and reads it line by line.
- Skips blank lines.
- Splits each line on ; into 5 parts.
- Ignores lines that do not have exactly 5 parts.
- Builds a dictionary per order.
- Converts quantity to int and price to float.
- If the file does not exist, creates an empty one.
- Any other error is printed instead of crashing.

**Example line:** ORD-101;Ali;Mocha;2;15.5

In [25]:
def load_data():
   
    records = []
    try:
        file = open(DATA_FILE, "r")
        lines = file.readlines()
        file.close()

        for line in lines:
            line = line.strip()
            if line:
                parts = line.split(";")
                if len(parts) == 5:
                    record = {
                        "order_id": parts[0],
                        "customer_name": parts[1],
                        "item_name": parts[2],
                        "quantity": int(parts[3]),
                        "total_price": float(parts[4])
                    }
                    records.append(record)
    except FileNotFoundError:
        file = open(DATA_FILE, "w")
        file.close()
    except Exception as e:
        print("Failed to load records correctly:", e)

    return records

## 3. save_data(records): write orders to file

Called after every change so nothing is lost.

- Opens the file in write mode ("w").
- This replaces the old file contents.
- Writes one order per line.
- Fields are joined with ;
- This is the reverse of load_data().
- File errors are caught and printed.

In [26]:
def save_data(records):
    
    try:
        file = open(DATA_FILE, "w")
        for record in records:
            line = f"{record['order_id']};{record['customer_name']};{record['item_name']};{record['quantity']};{record['total_price']}\n"
            file.write(line)
        file.close()
    except Exception as e:
        print("Failed to write data to file:", e)

## 4. user_login(): staff authentication

Protects the portal with a username and password.

- The user gets 3 attempts.
- Valid login is admin / admin123.
- Correct credentials: prints a welcome and returns True.
- Wrong credentials: reduces the attempts left.
- After 3 failures: access is denied and returns False.

In [1]:
def user_login():
    attempts = 3
    while attempts > 0:
        username = input("Enter username: ").strip()
        password = input("Enter password: ").strip()

        if username == "admin" and password == "admin123":
            print("\n Access Granted! Welcome to Spill the Beans Portal.")
            return True
        else:
            attempts -= 1
            print(f"Invalid credentials. Attempts remaining: {attempts}")

    print("\n Maximum login attempts exceeded. Access denied.")
    return False

## 5. add_record(records): place a new order

Asks the user for details and adds a new order.

- Asks for an Order ID first.
- Stops if the ID already exists (case-insensitive).
- Asks for customer name and food item.
- Quantity must be a whole number above 0.
- Unit price must be a number, not negative.
- Invalid input repeats the question.
- Total price = quantity x unit price.
- Adds the order to the list and saves it.
- Prints a confirmation with the total.

**Note:** only the total is stored, not the unit price.

In [17]:
def add_record(records):
    
    order_id = input("Enter Order ID (e.g., ORD-101): ").strip()

    for record in records:
        if record["order_id"].lower() == order_id.lower():
            print("Order ID already exists. Please use a unique ID.")
            return

    customer_name = input("Enter Customer Name: ").strip()
    item_name = input("Enter Food Item Name: ").strip()

    while True:
        try:
            quantity = int(input("Enter Quantity (should be integer): "))
            if quantity > 0:
                break
            else:
                print("Quantity must be greater than 0.")
        except ValueError:
            print("Invalid input! Please enter a valid whole number for quantity.")

    while True:
        try:
            unit_price = float(input("Enter Unit Price ($): "))
            if unit_price >= 0:
                break
            else:
                print("Price cannot be negative.")
        except ValueError:
            print("Invalid input! Please enter a numerical price value.")

    total_price = quantity * unit_price

    new_order = {
        "order_id": order_id,
        "customer_name": customer_name,
        "item_name": item_name,
        "quantity": quantity,
        "total_price": total_price
    }

    records.append(new_order)
    save_data(records)
    print(f"\n Order '{order_id}' placed successfully for ${total_price:.2f}!")

## 6. search_record(records): find orders

Looks up orders by Order ID or customer name.

- Warns if there are no orders on file.
- Converts the search text to lowercase.
- Matches orders where the text appears in the ID or name.
- Partial matches work (e.g. ord or ali).
- Matches are shown in an aligned table.
- Prints a message if nothing matches.

In [24]:
def search_record(records):
    if not records:
        print("No order records currently on file.")
        return

    query = input("Enter Order ID or Customer Name to search: ").strip().lower()
    matches = []

    for record in records:
        if (query in record["order_id"].lower()) or (query in record["customer_name"].lower()):
            matches.append(record)

    if matches:
        print(f"\n Found {len(matches)} matching order(s):")
        print(f"{'Order ID':<10}| {'Customer':<18}| {'Item':<15}| {'Qty':<5}| {'Total ($)':<8}")
        for record in matches:
            print(f"{record['order_id']:<10} | {record['customer_name']:<18} | {record['item_name']:<15} | {record['quantity']:<5} | ${record['total_price']:<7.2f}")
    else:
        print("No matching orders found.")

## 7. update_record(records): edit an order

Changes the details of an existing order.

- Asks for the Order ID to update.
- Finds the matching order (case-insensitive).
- Asks for new name, item, quantity and unit price.
- Recalculates the total price.
- Saves the changes to the file.
- Prints a message if the ID is not found.

**Note:** the Order ID itself cannot be changed.

In [19]:
def update_record(records):
    order_id = input("Enter Order ID to update: ").strip().lower()

    for record in records:
        if record["order_id"].lower() == order_id:
            record["customer_name"] = input("Enter new Customer Name: ").strip()
            record["item_name"] = input("Enter new Food Item Name: ").strip()
            record["quantity"] = int(input("Enter new Quantity: "))
            unit_price = float(input("Enter new Unit Price ($): "))
            
            record["total_price"] = record["quantity"] * unit_price
            save_data(records)
            print(f"\n Order '{record['order_id']}' updated successfully!")
            return

    print(f"Order ID '{order_id}' not found.")

## 8. delete_record(records): remove an order

Deletes one order permanently.

- Asks for the Order ID to delete.
- Finds the matching order (case-insensitive).
- Removes it from the list.
- Saves the updated list to the file.
- Prints a message if the ID is not found.

In [20]:
def delete_record(records):
    order_id = input("Enter Order ID to delete: ").strip().lower()

    for record in records:
        if record["order_id"].lower() == order_id:
            records.remove(record)
            save_data(records)
            print(f"\n Order '{order_id}' deleted successfully.")
            return

    print(f"Order ID '{order_id}' not found.")

## 9. generate_report(records): sales summary

Prints an overview of all orders.

**Summary figures**

- Total number of orders.
- Total items sold.
- Gross revenue (sum of all totals).
- Average order value.
- The highest-value order and its customer.

**Then:** a list of every order.

If there are no orders, it prints a message and stops.

In [2]:
def generate_report(records):
    
    print("\nRestaurant Sales & Orders Report")
    if not records:
        print("No order data available to generate a report.")
        return

    total_orders = len(records)
    
    total_revenue = 0.0
    total_items_sold = 0
    for record in records:
        total_revenue += record["total_price"]
        total_items_sold += record["quantity"]

    avg_order_value = total_revenue / total_orders

    highest_order = records[0]
    for record in records:
        if record["total_price"] > highest_order["total_price"]:
            highest_order = record

    print(f"Total Orders Processed : {total_orders}")
    print(f"Total Items Sold       : {total_items_sold}")
    print(f"Gross Revenue          : ${total_revenue:.2f}")
    print(f"Average Order Value    : ${avg_order_value:.2f}")
    print(f"Highest Value Order    : {highest_order['order_id']} (${highest_order['total_price']:.2f} by {highest_order['customer_name']})")
    
    print("\nAll Customer Orders")
    for record in records:
        print(f"ID: {record['order_id']} ; Customer: {record['customer_name']} ; Item: {record['item_name']} ; Qty: {record['quantity']} ; Total: ${record['total_price']:.2f}")

## 10. main(): the program loop

Ties everything together.

- Loads saved orders from the file.
- Runs the login. Exits if it fails.
- Shows a menu repeatedly until the user quits.

Any other input shows a reminder to choose 1 to 6.

The last line, main(), starts the program.

In [27]:
def main():
    records = load_data()

    if not user_login():
        return

    while True:
        
        print("1. Place New Order")
        print("2. Search Order Record")
        print("3. Update Existing Order")
        print("4. Delete Order Record")
        print("5. Generate Sales Report")
        print("6. Exit Portal")

        choice = input("Select an option (1-6): ").strip()

        if choice == '1':
            add_record(records)
        elif choice == '2':
            search_record(records)
        elif choice == '3':
            update_record(records)
        elif choice == '4':
            delete_record(records)
        elif choice == '5':
            generate_report(records)
        elif choice == '6':
            save_data(records)
            print("Exiting the portal. Data entered, if any, is saved.")
            break
        else:
            print("Please only enter a choice from 1 to 6.")


main()

Enter username:  admin
Enter password:  admin123



 Access Granted! Welcome to the Restaurant Portal.
1. Place New Order
2. Search Order Record
3. Update Existing Order
4. Delete Order Record
5. Generate Sales Report
6. Exit Portal


Select an option (1-6):  1
Enter Order ID (e.g., ORD-101):  ORD-000
Enter Customer Name:  Sumaiya Kashif
Enter Food Item Name:  Mocha Latte
Enter Quantity (should be integer):  1
Enter Unit Price ($):  7



 Order 'ORD-000' placed successfully for $7.00!
1. Place New Order
2. Search Order Record
3. Update Existing Order
4. Delete Order Record
5. Generate Sales Report
6. Exit Portal


Select an option (1-6):  1
Enter Order ID (e.g., ORD-101):  ORD-001
Enter Customer Name:  Aqsa Tabassum
Enter Food Item Name:  Hazelnut Frappe
Enter Quantity (should be integer):  2
Enter Unit Price ($):  10



 Order 'ORD-001' placed successfully for $20.00!
1. Place New Order
2. Search Order Record
3. Update Existing Order
4. Delete Order Record
5. Generate Sales Report
6. Exit Portal


... (12 outputs left out)

Select an option (1-6):  5



Restaurant Sales & Orders Report
Total Orders Processed : 3
Total Items Sold       : 5
Gross Revenue          : $30.00
Average Order Value    : $10.00
Highest Value Order    : ORD-001 ($18.00 by Aqsa Tabassum)

All Customer Orders
ID: ORD-000 ; Customer: Sumaiya Kashif ; Item: Mocha Latte ; Qty: 1 ; Total: $7.00
ID: ORD-001 ; Customer: Aqsa Tabassum ; Item: Lava Cake ; Qty: 3 ; Total: $18.00
ID: ORD-102 ; Customer: Alina Sultan ; Item: Iced Vanilla Bean ; Qty: 1 ; Total: $5.00
1. Place New Order
2. Search Order Record
3. Update Existing Order
4. Delete Order Record
5. Generate Sales Report
6. Exit Portal


Select an option (1-6):  6


Exiting the portal. Data entered, if any, is saved.
